
#### 03 — Vocabulary and Bag of Words

##### Purpose

This notebook introduces the first numerical representation of support-ticket text: Bag of Words (BoW).

The notebook:

- reads the NLP-preprocessed dataset created by Notebook 02
- creates a reproducible train/validation/test split
- prevents identical cleaned text from leaking across splits
- persists the split assignment for reuse across future models
- learns a vocabulary from training data only
- transforms ticket text into Bag-of-Words vectors
- examines the resulting sparse feature matrix
- verifies that validation and test data do not influence the learned vocabulary

The notebook is independently runnable and does not depend on another notebook's execution state.

##### 1. Architecture


 NLP pipeline has reached this stage:

``` text

Raw Ticket Text
       ↓
02 Text Cleaning
       ↓
nlp_preprocessed_tickets
       ↓
03 Train / Validation / Test Split
       ↓
Training Text
       ↓
CountVectorizer.fit()
       ↓
Vocabulary
       ↓
Bag-of-Words Features

```


##### 2. Technologies

- Python
- PySpark
- pandas
- scikit-learn
- CountVectorizer
- Delta Lake
- Unity Catalog

##### 3. Imports

In [0]:
import numpy as np
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.window import Window

from sklearn.feature_extraction.text import CountVectorizer

from src.project_config import (
    NLP_CLEAN_TABLE,
    MODELING_TABLE,
    TICKET_ID_COL,
    TEXT_COL,
    TARGET_COL,
    CLEAN_TEXT_COL,
    TOKENS_COL,
    TOKEN_COUNT_COL,
    SPLIT_COL,
    EXPECTED_CATEGORIES,
    TRAIN_SIZE,
    VALIDATION_SIZE,
    TEST_SIZE,
    RANDOM_SEED,
    BOW_MAX_FEATURES,
)

##### 4. Verify Project Configuration

In [0]:
print(f"Input table    : {NLP_CLEAN_TABLE}")
print(f"Modeling table : {MODELING_TABLE}")

print(
    "Split ratios   : "
    f"train={TRAIN_SIZE:.0%}, "
    f"validation={VALIDATION_SIZE:.0%}, "
    f"test={TEST_SIZE:.0%}"
)

print(f"Random seed    : {RANDOM_SEED}")

##### 5. Load the NLP-Preprocessed Dataset

In [0]:
nlp_df = spark.table(
    NLP_CLEAN_TABLE
)

In [0]:
display(
    nlp_df.limit(20)
)

In [0]:
source_row_count = nlp_df.count()

print(
    f"Input row count: "
    f"{source_row_count:,}"
)

In [0]:
if source_row_count == 0:
    raise ValueError(
        f"Input table contains no records: "
        f"{NLP_CLEAN_TABLE}"
    )

##### 6. Validate Input Contract

In [0]:
required_columns = {
    TICKET_ID_COL,
    TEXT_COL,
    CLEAN_TEXT_COL,
    TOKENS_COL,
    TOKEN_COUNT_COL,
    TARGET_COL,
}

In [0]:
available_columns = set(
    nlp_df.columns
)

missing_columns = (
    required_columns
    - available_columns
)

if missing_columns:
    raise ValueError(
        "Missing required NLP columns: "
        f"{sorted(missing_columns)}"
    )

print(
    "Input schema validation passed."
)

##### 7. Validate Target Categories

In [0]:
actual_categories = {
    row[TARGET_COL]
    for row in (
        nlp_df
        .select(TARGET_COL)
        .distinct()
        .collect()
    )
}

In [0]:
expected_categories = set(
    EXPECTED_CATEGORIES
)

In [0]:
if actual_categories != expected_categories:
    raise ValueError(
        "Dataset categories do not match "
        "the configured project categories.\n"
        f"Expected: {sorted(expected_categories)}\n"
        f"Actual:   {sorted(actual_categories)}"
    )

print(
    "Category validation passed."
)

##### 8. Why Split Before Building the Vocabulary?


Suppose we have:

- Training tickets
- internet slow
- billing problem
- Test ticket
- router failure

If we fit CountVectorizer using all three tickets, the vocabulary might become:

- billing
- failure
- internet
- problem
- router
- slow

But router and failure came from the test set.

The model indirectly learned something about unseen test data before evaluation.

That is data leakage.

Instead:

``` text

Training text
      ↓
CountVectorizer.fit()
      ↓
Vocabulary learned
      ↓
Validation/Test
      ↓
CountVectorizer.transform()

```

The test data must remain unseen during fitting.

##### 9. Check for Conflicting Duplicate Text

In [0]:
conflicting_text_df = (
    nlp_df
    .groupBy(
        CLEAN_TEXT_COL
    )
    .agg(
        F.countDistinct(
            TARGET_COL
        ).alias(
            "category_count"
        )
    )
    .filter(
        F.col("category_count") > 1
    )
)

In [0]:
conflicting_text_count = (
    conflicting_text_df.count()
)

print(
    "Clean texts with conflicting labels: "
    f"{conflicting_text_count:,}"
)

In [0]:
if conflicting_text_count > 0:
    display(
        conflicting_text_df
    )

    raise ValueError(
        "Identical cleaned text is associated "
        "with multiple target categories."
    )

print(
    "Duplicate-label consistency validation passed."
)

##### 10. Why Identical Text Must Stay in the Same Split

Your dataset contains repeated examples such as:

- router is not working
- router is not working
- router is not working

If one copy appears in training and another identical copy appears in test:

TRAIN
router is not working

TEST
router is not working

the test example is no longer genuinely unseen.

That can make evaluation look better than the model really is.

Therefore we assign the split at the level of:

clean_text + category

rather than independently assigning every row.

##### 11. Build Unique Text Groups

In [0]:
text_groups_df = (
    nlp_df
    .select(
        CLEAN_TEXT_COL,
        TARGET_COL,
    )
    .distinct()
)

In [0]:
unique_group_count = (
    text_groups_df.count()
)

print(
    f"Unique text groups: "
    f"{unique_group_count:,}"
)

##### 12. Count Unique Groups per Category

In [0]:
group_count_by_category_df = (
    text_groups_df
    .groupBy(
        TARGET_COL
    )
    .count()
    .orderBy(
        TARGET_COL
    )
)

In [0]:
display(
    group_count_by_category_df
)

##### 13. Create a Deterministic Group Ordering

In [0]:
text_groups_df = (
    text_groups_df
    .withColumn(
        "_split_hash",
        F.xxhash64(
            F.concat_ws(
                "||",
                F.col(TARGET_COL),
                F.col(CLEAN_TEXT_COL),
                F.lit(
                    str(RANDOM_SEED)
                ),
            )
        ),
    )
)

##### 14. Rank Groups Within Each Category

In [0]:
category_window = (
    Window
    .partitionBy(
        TARGET_COL
    )
    .orderBy(
        F.col("_split_hash"),
        F.col(CLEAN_TEXT_COL),
    )
)

In [0]:
category_count_window = (
    Window
    .partitionBy(
        TARGET_COL
    )
)

In [0]:
text_groups_df = (
    text_groups_df
    .withColumn(
        "_group_rank",
        F.row_number().over(
            category_window
        ),
    )
    .withColumn(
        "_group_count",
        F.count("*").over(
            category_count_window
        ),
    )
)

##### 15. Calculate Relative Group Position

In [0]:
text_groups_df = (
    text_groups_df
    .withColumn(
        "_split_fraction",
        (
            F.col("_group_rank") - 1
        )
        / F.col("_group_count"),
    )
)

In [0]:
display(text_groups_df)

##### 16. Assign Dataset Split

In [0]:
train_boundary = TRAIN_SIZE

validation_boundary = (
    TRAIN_SIZE
    + VALIDATION_SIZE
)

In [0]:
text_groups_df = (
    text_groups_df
    .withColumn(
        SPLIT_COL,
        F.when(
            F.col("_split_fraction")
            < train_boundary,
            F.lit("train"),
        )
        .when(
            F.col("_split_fraction")
            < validation_boundary,
            F.lit("validation"),
        )
        .otherwise(
            F.lit("test")
        ),
    )
)

In [0]:
display(text_groups_df)

In [0]:
display(
    text_groups_df
    .select(
        CLEAN_TEXT_COL,
        TARGET_COL,
        SPLIT_COL,
    )
    .orderBy(
        TARGET_COL,
        SPLIT_COL,
    )
)

##### 17. Join Split Assignment Back to All Tickets

In [0]:
modeling_df = (
    nlp_df
    .join(
        text_groups_df.select(
            CLEAN_TEXT_COL,
            TARGET_COL,
            SPLIT_COL,
        ),
        on=[
            CLEAN_TEXT_COL,
            TARGET_COL,
        ],
        how="inner",
    )
)

In [0]:
modeling_row_count = (
    modeling_df.count()
)

if modeling_row_count != source_row_count:
    raise ValueError(
        "Row count changed unexpectedly "
        "while assigning dataset splits."
    )

print(
    "Split assignment row-count "
    "validation passed."
)

In [0]:
display(modeling_df)

##### 18. Inspect Split Distribution

In [0]:
split_distribution_df = (
    modeling_df
    .groupBy(
        SPLIT_COL
    )
    .count()
    .withColumn(
        "percentage",
        F.round(
            (
                F.col("count")
                / modeling_row_count
            )
            * 100,
            2,
        ),
    )
    .orderBy(
        SPLIT_COL
    )
)

In [0]:
display(
    split_distribution_df
)

##### 19. Inspect Split Distribution by Category

In [0]:
split_category_distribution_df = (
    modeling_df
    .groupBy(
        TARGET_COL,
        SPLIT_COL,
    )
    .count()
    .orderBy(
        TARGET_COL,
        SPLIT_COL,
    )
)


In [0]:
display(
    split_category_distribution_df
)

##### 20. Validate All Categories Are Represented

In [0]:
split_category_counts = (
    modeling_df
    .groupBy(
        SPLIT_COL
    )
    .agg(
        F.countDistinct(
            TARGET_COL
        ).alias(
            "category_count"
        )
    )
)

In [0]:
display(
    split_category_counts
)

In [0]:
expected_category_count = len(
    EXPECTED_CATEGORIES
)

In [0]:
invalid_split_count = (
    split_category_counts
    .filter(
        F.col("category_count")
        != expected_category_count
    )
    .count()
)

In [0]:
if invalid_split_count > 0:
    raise ValueError(
        "At least one dataset split does not "
        "contain all expected categories. "
        "Review the number of unique text groups "
        "per category."
    )

print(
    "Category coverage validation passed."
)

##### 21. Verify Duplicate Text Does Not Cross Splits

In [0]:
cross_split_duplicates_df = (
    modeling_df
    .groupBy(
        CLEAN_TEXT_COL
    )
    .agg(
        F.countDistinct(
            SPLIT_COL
        ).alias(
            "split_count"
        )
    )
    .filter(
        F.col("split_count") > 1
    )
)

In [0]:
cross_split_duplicate_count = (
    cross_split_duplicates_df.count()
)

print(
    "Clean texts appearing in multiple splits: "
    f"{cross_split_duplicate_count:,}"
)

In [0]:
if cross_split_duplicate_count > 0:
    raise ValueError(
        "Duplicate cleaned text was found "
        "across multiple dataset splits."
    )

print(
    "Cross-split duplicate validation passed."
)

##### 22. Persist the Modeling Dataset

In [0]:
modeling_output_df = (
    modeling_df
    .select(
        TICKET_ID_COL,
        TEXT_COL,
        CLEAN_TEXT_COL,
        TOKENS_COL,
        TOKEN_COUNT_COL,
        TARGET_COL,
        SPLIT_COL,
    )
)

In [0]:
(
    modeling_output_df
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true",
    )
    .saveAsTable(
        MODELING_TABLE
    )
)

In [0]:
print(
    f"Saved modeling dataset: "
    f"{MODELING_TABLE}"
)

In [0]:
%sql
select * from dbw_agentic_ai_dev.support_ticket_ai.nlp_modeling_dataset


##### 23. Verify Persisted Modeling Dataset

In [0]:
persisted_modeling_df = (
    spark.table(
        MODELING_TABLE
    )
)

In [0]:
persisted_modeling_count = (
    persisted_modeling_df.count()
)

if persisted_modeling_count != modeling_row_count:
    raise ValueError(
        "Persisted modeling dataset row count "
        "does not match the prepared dataset."
    )

print(
    "Modeling dataset persistence "
    "validation passed."
)

In [0]:
display(
    persisted_modeling_df
    .orderBy(
        SPLIT_COL,
        TARGET_COL,
        TICKET_ID_COL,
    )
)

##### 24. Create Train, Validation, and Test DataFrames

In [0]:
train_df = (
    persisted_modeling_df
    .filter(
        F.col(SPLIT_COL) == "train"
    )
)

validation_df = (
    persisted_modeling_df
    .filter(
        F.col(SPLIT_COL) == "validation"
    )
)

test_df = (
    persisted_modeling_df
    .filter(
        F.col(SPLIT_COL) == "test"
    )
)

In [0]:
train_count = train_df.count()
validation_count = validation_df.count()
test_count = test_df.count()

print(f"Train rows      : {train_count:,}")
print(f"Validation rows : {validation_count:,}")
print(f"Test rows       : {test_count:,}")

##### 25. Convert Modeling Text to pandas

In [0]:
train_pdf = (
    train_df
    .select(
        TICKET_ID_COL,
        CLEAN_TEXT_COL,
        TARGET_COL,
    )
    .toPandas()
)

validation_pdf = (
    validation_df
    .select(
        TICKET_ID_COL,
        CLEAN_TEXT_COL,
        TARGET_COL,
    )
    .toPandas()
)

test_pdf = (
    test_df
    .select(
        TICKET_ID_COL,
        CLEAN_TEXT_COL,
        TARGET_COL,
    )
    .toPandas()
)

In [0]:
print(
    "Train shape      :",
    train_pdf.shape,
)

print(
    "Validation shape :",
    validation_pdf.shape,
)

print(
    "Test shape       :",
    test_pdf.shape,
)

##### 26. What Is a Vocabulary?

Suppose the training corpus contains:

- internet connection problem
- billing payment problem
- internet service problem

Unique terms are:

- internet
- connection
- problem
- billing
- payment
- service

That collection is the vocabulary.

Conceptually:

- internet   → feature 0
- connection → feature 1
- problem    → feature 2
- billing    → feature 3
- payment    → feature 4
- service    → feature 5

The feature number itself has no numerical meaning.

It simply identifies a column.

##### 27. What Is Bag of Words?

For vocabulary: [internet, connection, problem, billing, payment, service]

ticket: internet connection problem

becomes: [1, 1, 1, 0, 0, 0]

while: billing payment problem

becomes: [0, 0, 1, 1, 1, 0]

The value represents the number of occurrences of each vocabulary term.

Bag of Words therefore preserves:

- which words occur
- how many times they occur

but generally loses:

- word order
- context
- semantic meaning

##### 28. Create the CountVectorizer

clean_text column is already:

- lowercase
- whitespace normalized
- punctuation normalized

and Notebook 02 already established whitespace tokenization.

Therefore configure CountVectorizer to use the same whitespace tokenization rather than applying a different tokenizer.

In [0]:
bow_vectorizer = CountVectorizer(
    lowercase=False,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    max_features=BOW_MAX_FEATURES,
)

In [0]:
bow_vectorizer

##### 29. Fit Vocabulary on Training Data Only

In [0]:
X_train_bow = (
    bow_vectorizer
    .fit_transform(
        train_pdf[
            CLEAN_TEXT_COL
        ]
    )
)

In [0]:
X_train_bow

##### 30. Transform Validation Data

In [0]:
X_validation_bow = (
    bow_vectorizer
    .transform(
        validation_pdf[
            CLEAN_TEXT_COL
        ]
    )
)

In [0]:
X_validation_bow

##### 31. Transform Test Data

In [0]:
X_test_bow = (
    bow_vectorizer
    .transform(
        test_pdf[
            CLEAN_TEXT_COL
        ]
    )
)

In [0]:
X_test_bow

##### 32. Inspect Learned Vocabulary

In [0]:
feature_names = (
    bow_vectorizer
    .get_feature_names_out()
)

In [0]:
feature_names

In [0]:
vocabulary_size = len(
    feature_names
)

print(
    f"Vocabulary size: "
    f"{vocabulary_size:,}"
)

In [0]:
if vocabulary_size == 0:
    raise ValueError(
        "CountVectorizer learned "
        "an empty vocabulary."
    )

In [0]:
vocabulary_df = pd.DataFrame(
    {
        "feature_index": np.arange(
            vocabulary_size
        ),
        "term": feature_names,
    }
)

display(
    vocabulary_df
)

# The vocabulary was learned only from training tickets.

##### 33. Inspect the Vocabulary Mapping

In [0]:
sorted_vocabulary = sorted(
    bow_vectorizer.vocabulary_.items(),
    key=lambda item: item[1],
)

for term, index in sorted_vocabulary[:30]:
    print(
        f"{index:>3} -> {term}"
    )

##### 34. Inspect Feature Matrix Shapes

In [0]:
print(
    "Train BoW shape      :",
    X_train_bow.shape,
)

print(
    "Validation BoW shape :",
    X_validation_bow.shape,
)

print(
    "Test BoW shape       :",
    X_test_bow.shape,
)

In [0]:
if not (
    X_train_bow.shape[1]
    == X_validation_bow.shape[1]
    == X_test_bow.shape[1]
):
    raise ValueError(
        "Bag-of-Words feature dimensions "
        "do not match across datasets."
    )

print(
    "BoW feature-dimension "
    "validation passed."
)

##### 35. Sparse Matrix Representation

In [0]:
print(
    type(
        X_train_bow
    )
)

You should see a SciPy sparse matrix type.

Why sparse?

Suppose our vocabulary contains: 1,000 words

but one ticket contains only: 5 words

Most feature values are therefore: 0

Instead of physically storing: [0,0,0,0,1,0,0,0,...]

a sparse matrix stores mostly the non-zero information. This saves memory.

##### 36. Calculate Matrix Sparsity

In [0]:
total_cells = (
    X_train_bow.shape[0]
    * X_train_bow.shape[1]
)

In [0]:
non_zero_values = (
    X_train_bow.nnz
)

In [0]:
sparsity = (
    1
    - (
        non_zero_values
        / total_cells
    )
)

In [0]:
print(
    f"Total matrix cells : "
    f"{total_cells:,}"
)

print(
    f"Non-zero values    : "
    f"{non_zero_values:,}"
)

print(
    f"Sparsity           : "
    f"{sparsity:.2%}"
)

##### 37. Inspect One Training Ticket

In [0]:
sample_index = 0

sample_ticket = (
    train_pdf
    .iloc[
        sample_index
    ]
)

In [0]:
print(
    "Ticket ID:"
)

print(
    sample_ticket[
        TICKET_ID_COL
    ]
)

print(
    "\nClean text:"
)

print(
    sample_ticket[
        CLEAN_TEXT_COL
    ]
)

print(
    "\nCategory:"
)

print(
    sample_ticket[
        TARGET_COL
    ]
)

##### 38. Inspect Its Bag-of-Words Vector

In [0]:
sample_vector = (
    X_train_bow[
        sample_index
    ]
)

In [0]:
sample_vector

In [0]:
sample_dense = (
    sample_vector
    .toarray()
    .ravel()
)

In [0]:
sample_dense

In [0]:
non_zero_indices = (
    sample_dense
    .nonzero()[0]
)

In [0]:
non_zero_indices

In [0]:
sample_features_df = pd.DataFrame(
    {
        "term": (
            feature_names[
                non_zero_indices
            ]
        ),
        "count": (
            sample_dense[
                non_zero_indices
            ]
        ),
    }
)

In [0]:
display(
    sample_features_df
    .sort_values(
        "term"
    )
)

##### 39. Demonstrate Word Counts

In [0]:
count_demo_text = [
    "refund refund refund billing"
]

In [0]:
count_demo_vector = (
    bow_vectorizer
    .transform(
        count_demo_text
    )
)

In [0]:
count_demo_vector

In [0]:
count_demo_dense = (
    count_demo_vector
    .toarray()
    .ravel()
)

In [0]:
count_demo_dense

In [0]:
count_demo_indices = (
    count_demo_dense
    .nonzero()[0]
)

In [0]:
count_demo_indices

In [0]:
count_demo_df = pd.DataFrame(
    {
        "term": (
            feature_names[
                count_demo_indices
            ]
        ),
        "count": (
            count_demo_dense[
                count_demo_indices
            ]
        ),
    }
)

In [0]:
display(
    count_demo_df
)

##### 40. Demonstrate an Unknown Word

In [0]:
unknown_demo_text = [
    "internet supercalifragilistic"
]

In [0]:
unknown_demo_vector = (
    bow_vectorizer
    .transform(
        unknown_demo_text
    )
)

##### 41. Verify Vocabulary Comes Only From Training Data

In [0]:
training_vocabulary = set(
    feature_names
)

In [0]:
validation_words = set(
    " ".join(
        validation_pdf[
            CLEAN_TEXT_COL
        ]
    ).split()
)

test_words = set(
    " ".join(
        test_pdf[
            CLEAN_TEXT_COL
        ]
    ).split()
)

In [0]:
validation_only_words = (
    validation_words
    - training_vocabulary
)

In [0]:
test_only_words = (
    test_words
    - training_vocabulary
)

In [0]:
print(
    "Validation words not in "
    "training vocabulary:"
)

print(
    sorted(
        validation_only_words
    )
)

print(
    "\nTest words not in "
    "training vocabulary:"
)

print(
    sorted(
        test_only_words
    )
)

##### 42. Bag-of-Words Data Flow


``` text

Training clean_text
        ↓
CountVectorizer.fit_transform()
        ↓
X_train_bow


Validation clean_text
        ↓
CountVectorizer.transform()
        ↓
X_validation_bow


Test clean_text
        ↓
CountVectorizer.transform()
        ↓
X_test_bow

```

In [0]:
y_train = (
    train_pdf[
        TARGET_COL
    ]
)

y_validation = (
    validation_pdf[
        TARGET_COL
    ]
)

y_test = (
    test_pdf[
        TARGET_COL
    ]
)

In [0]:
print(
    "X_train:",
    X_train_bow.shape,
)

print(
    "y_train:",
    y_train.shape,
)

##### 43. Bag of Words vs Original Dataset

Before NLP representation:

#####ticket_text
-------------------------------
Router is not working

After cleaning:

##### clean_text
-------------------------------
router is not working

After vocabulary learning:

router  → column 42
is      → column 18
not     → column 28
working → column 55

After Bag of Words:

[0,0,0,...1,...1,...1,...1,...]

Now the text has become numerical features that a traditional ML algorithm can use.

##### 44. What Bag of Words Learns — and Does Not Learn

Bag of Words captures:

word occurrence
word frequency

It does not understand that: refund and reimbursement may have similar meanings.

It also does not naturally understand word order.

For example:

service is not working

and:

service is working not

could contain essentially the same Bag-of-Words counts.

This is a major limitation.

##### 45. Why We Need TF-IDF Next

Bag of Words treats frequent words based primarily on their counts.

For example:

- is
- my
- the
- service

may appear frequently across many tickets.

But some words may be much more informative:

- refund
- router
- password
- cancel

TF-IDF improves Bag of Words by asking:

How important is this word to this document relative to the entire corpus?

So the progression is:

``` text

Bag of Words
    ↓
counts words
    ↓
limitation:
all counts treated similarly
    ↓
TF-IDF
    ↓
weights informative words

```

##### 46. Final Validation

In [0]:
assert train_count > 0
assert validation_count > 0
assert test_count > 0

assert vocabulary_size > 0

assert (
    X_train_bow.shape[0]
    == train_count
)

assert (
    X_validation_bow.shape[0]
    == validation_count
)

assert (
    X_test_bow.shape[0]
    == test_count
)

assert (
    X_train_bow.shape[1]
    == X_validation_bow.shape[1]
    == X_test_bow.shape[1]
)

print(
    "Final Bag-of-Words "
    "validation passed."
)

##### 47. Modeling Dataset Contract

Notebook 03 creates this reusable table: dbw_agentic_ai_dev.support_ticket_ai.nlp_modeling_dataset

Contract:

- ticket_id
- ticket_text
- clean_text
- tokens
- token_count
- category
- dataset_split

The important new field is: dataset_split

with values:

- train
- validation
- test

Later notebooks should use this existing assignment instead of creating their own random splits.

##### 48. Production Design Decisions

This notebook follows several important production principles.

- The NLP preprocessing table is read from Unity Catalog rather than notebook state.
- Train/validation/test assignment is reproducible.
- Identical cleaned text stays in one split to reduce leakage.
- Conflicting labels for identical text fail validation.
- Split assignments are persisted for reuse across modeling approaches.
- The vocabulary is fitted using training data only.
- Validation and test data call only transform().
- Bag-of-Words matrices remain sparse.
- The same fitted vectorizer defines the feature space for train, validation, and test datasets.
- Feature engineering remains separate from model training.

##### Key Learnings

1. Vocabulary - A vocabulary is the collection of unique terms learned from the training corpus. word → feature position.The feature index itself has no mathematical magnitude.

2. Bag of Words - Bag of Words converts text into word-count features.

``` text

text
 ↓
vocabulary
 ↓
word counts
 ↓
numerical vector

```

3. fit() vs transform() 

- fit() -> learns something
- transform() -> applies what was learned

For CountVectorizer:

``` text

fit()
    ↓
learn vocabulary

transform()
    ↓
use that vocabulary

```

4. Training Only -The vocabulary must be learned from training data only.

- Train      → fit + transform
- Validation → transform
- Test       → transform

This prevents data leakage.

5. Sparse Matrices -Text typically produces thousands of possible features while each individual ticket contains only a small subset. Therefore Bag-of-Words matrices are naturally sparse.

6. Duplicate Leakage - Identical text should not appear in both training and test sets because the test example would no longer be truly unseen.

7. Persistent Split Contract - By saving dataset_split, later approaches can be compared fairly using the same train/validation/test records.

##### Conclusion

Notebook 03 takes the cleaned text produced by Notebook 02 and performs the first major NLP conversion:

``` text 

Human Language
      ↓
Clean Text
      ↓
Train / Validation / Test Split
      ↓
Training Vocabulary
      ↓
Bag-of-Words Features
      ↓
Numerical ML Input

```

We now understand how: "router is not working"

can eventually become something like: [0, 1, 0, 0, 1, 1, 0, ..., 1]

This is the bridge between language and traditional machine learning.

We have not trained a classifier yet.

That is intentional.

First we are understanding how text becomes features.

##### Next Notebook

##### 04_tfidf_features

Notebook 04 will start from the limitation of Bag of Words:

``` text

BoW
 ↓
How many times does a word occur?

TF-IDF
 ↓
How important is that word?

```

We will reuse the exact same persisted:

train
validation
test

assignments and learn TF-IDF statistics from training data only.